# 🚀 Google Colab Ollama Server cho Dự Án Convert_File_V2
Notebook này thiết lập máy chủ dịch thuật **Ollama** chạy mô hình **Qwen2.5:7B-Instruct** trên Google Colab với GPU T4 miễn phí và mở **Cloudflare Tunnel** công khai để máy Local kết nối và dịch tự động.

### ⚠️ Lưu ý quan trọng:
1. Vào **Runtime (Thời gian chạy)** -> **Change runtime type (Thay đổi loại thời gian chạy)** -> Chọn **T4 GPU**.
2. Chạy lần lượt từng cell bên dưới từ trên xuống dưới.

In [ ]:
# Bước 1: Cài đặt Ollama và Cloudflared
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# Bước 2: Khởi động Ollama Service ngầm & Tải mô hình Qwen 2.5 7B
import subprocess
import time

print("Khởi động Ollama service ngầm...")
ollama_proc = subprocess.Popen(["ollama", "serve"])
time.sleep(3)

print("Tải mô hình qwen2.5:7b-instruct (khoảng 4.7 GB)...")
!ollama pull qwen2.5:7b-instruct

In [ ]:
# Bước 3: Mở Cloudflare Tunnel và lấy URL public kết nối về máy Local
import subprocess
import re

print("Đang kết nối Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print("=" * 65)
        print("🎉 MÁY CHỦ DỊCH THUẬT OLLAMA ĐÃ SẴN SÀNG!")
        print(f"👉 URL KẾT NỐI: {public_url}")
        print("Sao chép link trên và dán vào run_cli.py (Tùy chọn [15]) tại máy Local.")
        print("=" * 65)
        break

# Giữ cell này chạy liên tục để duy trì tunnel
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    tunnel_proc.kill()
    ollama_proc.kill()